### 1.Load and clean the flight dataset (handle missing/invalid delay times).

In [7]:
import pandas as pd
df = pd.read_csv("Flight_dataset.csv")
df.head()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27
0,2015-01-01,NK,195,MCO,FLL,2147,2143.0,-4.0,15.0,2158.0,...,63.0,62.0,40.0,177.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01,NK,197,LGA,FLL,1050,1104.0,14.0,20.0,1124.0,...,194.0,179.0,150.0,1076.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01,NK,198,FLL,MCO,700,712.0,12.0,19.0,731.0,...,57.0,61.0,32.0,177.0,0.0,0.0,16.0,0.0,0.0,NaN
3,2015-01-01,NK,199,IAH,LAS,2240,2251.0,11.0,8.0,2259.0,...,196.0,176.0,164.0,1222.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-01,NK,200,IAH,ORD,623,620.0,-3.0,15.0,635.0,...,152.0,140.0,115.0,925.0,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.columns = df.columns.str.strip().str.lower()

In [9]:
df['fl_date'] = pd.to_datetime(df['fl_date'])

In [10]:
df['dep_delay'] = df['dep_delay'].fillna(0)
df['arr_delay'] = df['arr_delay'].fillna(0)

In [11]:
df = df[df['dep_delay'] > -100]

### 2.Identify delay patterns based on day of the week.

In [13]:
df['day_of_week'] = df['fl_date'].dt.day_name()
delay_by_day = df.groupby('day_of_week')['dep_delay'].mean().sort_values(ascending=False)
delay_by_day

day_of_week
Monday       10.617359
Thursday      9.822929
Friday        9.338719
Sunday        9.257782
Tuesday       9.006004
Wednesday     8.541097
Saturday      7.734767
Name: dep_delay, dtype: float64

### 3.Analyze the impact of weather on delays.

In [15]:
weather_impact = df[['dep_delay', 'weather_delay']].corr()
weather_impact

,dep_delay,weather_delay
dep_delay,1.000000,0.243532
weather_delay,0.243532,1.000000


In [16]:
weather_delay_total = df['weather_delay'].sum()
total_delay = df['dep_delay'].sum()

impact_percent = (weather_delay_total / total_delay) * 100
print("Weather Impact %:", impact_percent)

Weather Impact %: 5.771265739292724


### 4.Calculate overall on-time performance rate.

In [18]:
on_time = df[df['dep_delay'] <= 0].shape[0]
total = df.shape[0]
on_time_rate = (on_time / total)* 100
print("on-time perfomance:",on_time_rate)

on-time perfomance: 63.47157342252958


### 5.Find delay trends by time of day (morning, afternoon, evening).

In [20]:
def time_category(i):
    if i < 1200:
        return "morning"
    elif i < 1700:
        return "Afternoon"
    else:
        return "Evening"

df['time_of_day'] = df['crs_dep_time'].apply(time_category)
time_delay = df.groupby('time_of_day')['dep_delay'].mean()
time_delay

time_of_day
Afternoon    10.782681
Evening      14.013658
morning       4.789467
Name: dep_delay, dtype: float64

### 6.Detect seasonal trends in delay rates (monthly/quarterly).

### Monthly

In [23]:
df['month'] = df['fl_date'].dt.month
monthly_delay = df.groupby('month')['dep_delay'].mean()
monthly_delay

month
1      9.517399
2     11.329804
3      9.457096
4      7.654192
5      9.352639
6     13.744597
7     11.296039
8      9.840240
9      4.803558
10     4.958689
11     6.879811
12    11.594714
Name: dep_delay, dtype: float64

### Quarterly

In [25]:
df['quarter'] = df['fl_date'].dt.quarter
quarterly_delay = df.groupby('quarter')['dep_delay'].mean()
quarterly_delay

quarter
1    10.049976
2    10.287399
3     8.781742
4     7.804585
Name: dep_delay, dtype: float64